In [4]:
import numpy as np
import pandas as pd
import joblib
from tqdm import tqdm  # 进度条库

# 定义文件夹路径
folder_path_qf = "qf_models"
folder_path_kl = "kl_models"

# 权重定义
weights_qf = {'xgboost': 0.7 / 5, 'rf': 0.3 / 5}
weights_kl = {'xgboost': 0.5 / 5, 'rf': 0.5 / 5}

# 加载测试集
df_qf = pd.read_excel('test_set_new.xlsx', index_col=0)
X_test_qf = df_qf.drop(columns=['Precipitate Distribution', 'Yield_Strength', 'Tensile_Strength (UTS)', '追踪编号'])
y_test_qf = df_qf['Yield_Strength']

df_kl = pd.read_excel('test_set_new.xlsx', index_col=0)
X_test_kl = df_kl.drop(columns=['Precipitate Distribution', 'Yield_Strength', 'Tensile_Strength (UTS)', '追踪编号'])
y_test_kl = df_kl['Tensile_Strength (UTS)']

# 特征及其变化范围
feature_ranges = {
    'Interant d electrons': np.arange(3, 11, 1),
}

# 创建Excel文件以保存结果
output_file_qf = 'AAA_qf_multiple_samples_analysis.xlsx'
output_file_kl = 'AAA_kl_multiple_samples_analysis.xlsx'

# 分别保存qf和kl的结果
with pd.ExcelWriter(output_file_qf, engine='openpyxl') as writer_qf, pd.ExcelWriter(output_file_kl, engine='openpyxl') as writer_kl:
    # 对qf样本进行分析
    for sample_index in tqdm(range(len(X_test_qf)), desc="Processing qf samples", unit="sample"):
        sample_qf = X_test_qf.iloc[sample_index].copy()
        predictions_changes_qf = []

        for feature, ranges in feature_ranges.items():
            predicted_values = []
            for value in ranges:
                modified_sample_qf = sample_qf.copy()
                modified_sample_qf[feature] = value
                modified_sample_qf_df = pd.DataFrame([modified_sample_qf], columns=X_test_qf.columns)

                prediction_for_this_value_qf = np.zeros(1)
                for model_name, weight in weights_qf.items():
                    for i in range(1, 6):
                        model_path_qf = f'{folder_path_qf}/{model_name}{i}_new.pkl'
                        model_qf = joblib.load(model_path_qf)
                        prediction_for_this_value_qf += model_qf.predict(modified_sample_qf_df) * weight

                predicted_values.append(prediction_for_this_value_qf[0])
                predictions_changes_qf.append((value, prediction_for_this_value_qf[0]))

            # 计算最大变化范围
            max_change_qf = max(predicted_values) - min(predicted_values)
            message_qf = f"Sample {sample_index}: d_electron impact on Yield Strength = {max_change_qf:.2f}"
            if max_change_qf > 30:
                print(f"\033[91m{message_qf}  ⚠️ Significant Impact! ⚠️\033[0m")  # 高亮显示
            else:
                print(message_qf)

        # 保存结果到Excel
        results_df_qf = pd.DataFrame(predictions_changes_qf, columns=[feature, 'Predicted Yield Strength'])
        results_df_qf.to_excel(writer_qf, sheet_name=f'sample_{sample_index}', index=False)

    # 对kl样本进行分析
    for sample_index in tqdm(range(len(X_test_kl)), desc="Processing kl samples", unit="sample"):
        sample_kl = X_test_kl.iloc[sample_index].copy()
        predictions_changes_kl = []

        for feature, ranges in feature_ranges.items():
            predicted_values = []
            for value in ranges:
                modified_sample_kl = sample_kl.copy()
                modified_sample_kl[feature] = value
                modified_sample_kl_df = pd.DataFrame([modified_sample_kl], columns=X_test_kl.columns)

                prediction_for_this_value_kl = np.zeros(1)
                for model_name, weight in weights_kl.items():
                    for i in range(1, 6):
                        model_path_kl = f'{folder_path_kl}/{model_name}{i}_new.pkl'
                        model_kl = joblib.load(model_path_kl)
                        prediction_for_this_value_kl += model_kl.predict(modified_sample_kl_df) * weight

                predicted_values.append(prediction_for_this_value_kl[0])
                predictions_changes_kl.append((value, prediction_for_this_value_kl[0]))

            # 计算最大变化范围
            max_change_kl = max(predicted_values) - min(predicted_values)
            message_kl = f"Sample {sample_index}: d_electron impact on Tensile Strength = {max_change_kl:.2f}"
            if max_change_kl > 30:
                print(f"\033[91m{message_kl}  ⚠️ Significant Impact! ⚠️\033[0m")  # 高亮显示
            else:
                print(message_kl)

        # 保存结果到Excel
        results_df_kl = pd.DataFrame(predictions_changes_kl, columns=[feature, 'Predicted Tensile Strength'])
        results_df_kl.to_excel(writer_kl, sheet_name=f'sample_{sample_index}', index=False)

print("Multi-sample single variable analysis for qf and kl completed and results saved to Excel.")


Processing qf samples:   1%|          | 1/88 [00:03<04:56,  3.41s/sample]

Sample 0: d_electron impact on Yield Strength = 2.33


Processing qf samples:   2%|▏         | 2/88 [00:05<03:52,  2.70s/sample]

Sample 1: d_electron impact on Yield Strength = 1.26


Processing qf samples:   3%|▎         | 3/88 [00:07<03:27,  2.44s/sample]

...

Processing kl samples:  99%|█████████▉| 87/88 [02:46<00:01,  1.90s/sample]

Sample 86: d_electron impact on Tensile Strength = 11.57


Processing kl samples: 100%|██████████| 88/88 [02:48<00:00,  1.91s/sample]

Sample 87: d_electron impact on Tensile Strength = 40.04  ⚠️ Significant Impact! ⚠️


Multi-sample single variable analysis for qf and kl completed and results saved to Excel.


In [9]:
import numpy as np
import pandas as pd
import joblib
from tqdm import tqdm  # 进度条库

# 定义文件夹路径
folder_path_qf = "qf_models"
folder_path_kl = "kl_models"

# 权重定义
weights_qf = {'xgboost': 0.7 / 5, 'rf': 0.3 / 5}
weights_kl = {'xgboost': 0.5 / 5, 'rf': 0.5 / 5}

# 读取测试集
df_qf = pd.read_excel('train_set_new.xlsx', index_col=0)
df_kl = pd.read_excel('train_set_new.xlsx', index_col=0)

# 目标变量
X_test_qf = df_qf.drop(columns=['Precipitate Distribution', 'Yield_Strength', 'Tensile_Strength (UTS)', '追踪编号'])
y_test_qf = df_qf['Yield_Strength']
track_numbers_qf = df_qf['追踪编号']

X_test_kl = df_kl.drop(columns=['Precipitate Distribution', 'Yield_Strength', 'Tensile_Strength (UTS)', '追踪编号'])
y_test_kl = df_kl['Tensile_Strength (UTS)']
track_numbers_kl = df_kl['追踪编号']

# 选择要改变的特征及其变化范围
feature_ranges = {
    'Interant d electrons': np.arange(3, 11, 1),
}

# 计算屈服强度变化并保存
with pd.ExcelWriter('AAA_qf_multiple_single_variable_analysis_d_electrons.xlsx', engine='openpyxl') as writer:
    for feature, ranges in feature_ranges.items():
        all_results = []
        print(f"正在计算 {feature} 对屈服强度的影响...")
        for sample_index in tqdm(range(len(X_test_qf)), desc="处理样本"):
            sample = X_test_qf.iloc[sample_index].copy()
            track_number = track_numbers_qf.iloc[sample_index]
            for value in tqdm(ranges, desc=f"变化 {feature}", leave=False):
                modified_sample = sample.copy()
                modified_sample[feature] = value
                modified_sample_df = pd.DataFrame([modified_sample], columns=X_test_qf.columns)

                prediction_for_this_value = np.zeros(1)
                for model_name, weight in weights_qf.items():
                    for i in tqdm(range(1, 6), desc=f"计算 {model_name}", leave=False):
                        model_path = f'{folder_path_qf}/{model_name}{i}_new.pkl'
                        model = joblib.load(model_path)
                        prediction_for_this_value += model.predict(modified_sample_df) * weight

                all_results.append((track_number, sample_index, value, prediction_for_this_value[0]))

        # 结果保存到 Excel
        results_df = pd.DataFrame(all_results, columns=['Track Number', 'Sample Index', feature, 'Predicted Yield Strength'])
        sheet_name = f'{feature}_analysis'
        results_df.to_excel(writer, sheet_name=sheet_name, index=False)

print("✅ 屈服强度单变量分析完成，结果已保存到 AAA_qf_multiple_single_variable_analysis_d_electrons.xlsx")

# 计算抗拉强度变化并保存
with pd.ExcelWriter('AAA_kl_multiple_single_variable_analysis_d_electrons.xlsx', engine='openpyxl') as writer:
    for feature, ranges in feature_ranges.items():
        all_results = []
        print(f"正在计算 {feature} 对抗拉强度的影响...")
        for sample_index in tqdm(range(len(X_test_kl)), desc="处理样本"):
            sample = X_test_kl.iloc[sample_index].copy()
            track_number = track_numbers_kl.iloc[sample_index]
            for value in tqdm(ranges, desc=f"变化 {feature}", leave=False):
                modified_sample = sample.copy()
                modified_sample[feature] = value
                modified_sample_df = pd.DataFrame([modified_sample], columns=X_test_kl.columns)

                prediction_for_this_value = np.zeros(1)
                for model_name, weight in weights_kl.items():
                    for i in tqdm(range(1, 6), desc=f"计算 {model_name}", leave=False):
                        model_path = f'{folder_path_kl}/{model_name}{i}_new.pkl'
                        model = joblib.load(model_path)
                        prediction_for_this_value += model.predict(modified_sample_df) * weight

                all_results.append((track_number, sample_index, value, prediction_for_this_value[0]))

        # 结果保存到 Excel
        results_df = pd.DataFrame(all_results, columns=['Track Number', 'Sample Index', feature, 'Predicted Tensile Strength'])
        sheet_name = f'{feature}_analysis'
        results_df.to_excel(writer, sheet_name=sheet_name, index=False)

print("✅ 抗拉强度单变量分析完成，结果已保存到 AAA_kl_multiple_single_variable_analysis_d_electrons.xlsx")


正在计算 Interant d electrons 对屈服强度的影响...


处理样本:   0%|          | 0/739 [00:00<?, ?it/s]















































处理样本:   0%|          | 1/739 [00:02<25:27,  2.07s/it]















































处理样本:   0%|          | 2/739 [00:04<26:16,  2.14s/it]















































处理样本:   0%|          | 3/739 [00:06<26:10,  2.13s/it]















































处理样本:   1%|          | 4/739 [00:08<25:54,  2.12s/it]















































处理样本:   1%|          | 5/739 [00:10<25:47,  2.11s/it]















































处理样本:   1%|          | 6/739 [00:12<25:56,  2.12s/it]















































处理样本:   1%|          | 7/739 [00:14<25:45,  2.11s/it]















































处理样本:   1%|          | 8/739 [00:16<25:35,  2.10s/it]















































处理样本:   1%|          | 9/739 [00:18<25:27,  2.09s/it]















































✅ 屈服强度单变量分析完成，结果已保存到 AAA_qf_multiple_single_variable_analysis_d_electrons.xlsx
正在计算 Interant d electrons 对抗拉强度的影响...


处理样本:   0%|          | 0/739 [00:00<?, ?it/s]















































处理样本:   0%|          | 1/739 [00:02<25:00,  2.03s/it]















































处理样本:   0%|          | 2/739 [00:04<24:59,  2.03s/it]















































处理样本:   0%|          | 3/739 [00:06<24:49,  2.02s/it]















































处理样本:   1%|          | 4/739 [00:08<24:38,  2.01s/it]















































处理样本:   1%|          | 5/739 [00:10<24:34,  2.01s/it]















































处理样本:   1%|          | 6/739 [00:12<24:38,  2.02s/it]















































处理样本:   1%|          | 7/739 [00:14<24:33,  2.01s/it]















































处理样本:   1%|          | 8/739 [00:16<24:23,  2.00s/it]















































处理样本:   1%|          | 9/739 [00:18<24:18,  2.00s/it]















































✅ 抗拉强度单变量分析完成，结果已保存到 AAA_kl_multiple_single_variable_analysis_d_electrons.xlsx


In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import r2_score, mean_absolute_error, mean_absolute_percentage_error
import joblib
import matplotlib.pyplot as plt

# 定义文件夹路径
folder_path_qf = "qf_models"
# 权重定义
weights_qf = {'xgboost': 0.7 / 5, 'rf': 0.3 / 5}

# 加载测试集
df_qf = pd.read_excel('qf_models/train_set_new.xlsx', index_col=0)
X_test_qf = df_qf.drop(columns=['Precipitate Distribution', 'Yield_Strength', 'Tensile_Strength (UTS)', '追踪编号'])
y_test_qf = df_qf['Yield_Strength']
track_numbers = df_qf['追踪编号']
sample_index = 3
# 选择样本索引
sample_indices = [sample_index]

# 特征及其变化范围
feature_ranges = {
    # 'Calculated Yield Strength': np.arange(50, 301, 1),  # 已知
    # 'Grain Size': np.arange(5, 31, 0.5),  # 已知
    # 'Y fraction': np.arange(0.005, 0.04, 0.001),
    # 'Gd fraction': np.arange(0.005, 0.04, 0.001)
    'Interant d electrons':np.arange(3,11,1),
}

# 创建Excel文件以保存结果
with pd.ExcelWriter('qf_multiple_single_variable_analysis.xlsx', engine='openpyxl') as writer:
    for feature, ranges in feature_ranges.items():
        results = []
        for sample_index in sample_indices:
            sample = X_test_qf.iloc[sample_index].copy()
            track_number = track_numbers.iloc[sample_index]
            max_change = 0
            max_change_value = None
            max_change_prediction = None
            previous_prediction = None
            for value in ranges:
                modified_sample = sample.copy()
                modified_sample[feature] = value
                modified_sample_df = pd.DataFrame([modified_sample], columns=X_test_qf.columns)

                prediction_for_this_value = np.zeros(1)
                # 遍历每个模型进行预测
                for model_name, weight in weights_qf.items():
                    for i in range(1, 6):
                        model_path = f'{folder_path_qf}/{model_name}{i}_new.pkl'
                        model = joblib.load(model_path)
                        prediction_for_this_value += model.predict(modified_sample_df) * weight

                if previous_prediction is not None:
                    change = abs(prediction_for_this_value[0] - previous_prediction)
                    if change > max_change:
                        max_change = change
                        max_change_value = value
                        max_change_prediction = prediction_for_this_value[0]

                previous_prediction = prediction_for_this_value[0]

            results.append((track_number, max_change_value, max_change_prediction))

        # 将结果转换为 DataFrame 并保存到 Excel的不同sheet中
        results_df = pd.DataFrame(results, columns=['Track Number', f'{feature} with Max Change', 'Predicted Yield Strength'])
        sheet_name = f'{feature}_max_change'
        results_df.to_excel(writer, sheet_name=sheet_name, index=False)

        # 绘制直方图
        plt.figure(figsize=(10, 6))
        plt.hist(results_df[f'{feature} with Max Change'], bins=20, edgecolor='black')
        plt.xlabel(f'{feature} with Max Change')
        plt.ylabel('Frequency')
        plt.title(f'Histogram of {feature} with Max Change')
        plt.grid(True)
        plt.savefig(f'{feature}_max_change_histogram.png')
        plt.close()

print("Multi-feature single variable analysis completed and results saved to Excel and histograms.")


# 抗拉强度结果变化
import numpy as np
import pandas as pd
import joblib

# 定义文件夹路径
folder_path_kl = "kl_models"
# 权重定义
weights_kl = {'xgboost': 0.5 / 5, 'rf': 0.5 / 5}

# 加载测试集
df_kl = pd.read_excel('kl_models/test_set_new.xlsx', index_col=0)
X_test_kl = df_kl.drop(columns=['Precipitate Distribution', 'Yield_Strength', 'Tensile_Strength (UTS)', '追踪编号'])
y_test_kl = df_kl['Tensile_Strength (UTS)']

# 选择一个样本

# 创建Excel文件以保存结果
with pd.ExcelWriter('kl_multiple_single_variable_analysis.xlsx', engine='openpyxl') as writer:
    for feature, ranges in feature_ranges.items():
        predictions_changes = []
        for value in ranges:
            modified_sample = sample.copy()
            modified_sample[feature] = value
            modified_sample_df = pd.DataFrame([modified_sample], columns=X_test_kl.columns)
            prediction_for_this_value = np.zeros(1)
            # 遍历每个模型进行预测
            for model_name, weight in weights_kl.items():
                for i in range(1,6):
                    model_path = f'{folder_path_kl}/{model_name}{i}_new.pkl'
                    model = joblib.load(model_path)
                    prediction_for_this_value += model.predict(modified_sample_df) * weight

            predictions_changes.append((value, prediction_for_this_value[0]))

        # 将结果转换为 DataFrame 并保存到 Excel的不同sheet中
        results_df = pd.DataFrame(predictions_changes, columns=[feature, 'Predicted Tensile Strength'])
        results_df.to_excel(writer, sheet_name=feature, index=False)

print("Multi-feature single variable analysis for tensile strength completed and results saved to Excel.")


Multi-feature single variable analysis completed and results saved to Excel and histograms.
Multi-feature single variable analysis for tensile strength completed and results saved to Excel.


In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import r2_score, mean_absolute_error, mean_absolute_percentage_error
import joblib

# 定义文件夹路径
folder_path_qf = "qf_models"
# 权重定义
weights_qf = {'xgboost': 0.7 / 5, 'rf': 0.3 / 5}

# 加载测试集
df_qf = pd.read_excel('qf_models/test_set_new.xlsx', index_col=0)
X_test_qf = df_qf.drop(columns=['Precipitate Distribution',  'Yield_Strength', 'Tensile_Strength (UTS)', '追踪编号'])
y_test_qf = df_qf['Yield_Strength']

# 选择一个样本
sample_index = 8  # 根据需要更改样本索引
sample = X_test_qf.iloc[sample_index].copy()
# print(X_test_qf.iloc[sample_index,-5:],df_qf.iloc[sample_index]['Yield_Strength'])
# 特征及其变化范围
feature_ranges = {
    # 'Calculated Yield Strength': np.arange(50, 301, 1),  # 已知
    # 'Grain Size': np.arange(2, 31, 0.5),  # 已知
    # 'Y fraction': np.arange(0.005, 0.04, 0.001),
    # 'Gd fraction': np.arange(0.005, 0.04, 0.001)
    'Interant d electrons':np.arange(3,11,1),
}
# 创建Excel文件以保存结果
with pd.ExcelWriter('qf_multiple_single_variable_analysis.xlsx', engine='openpyxl') as writer:
    for feature, ranges in feature_ranges.items():
        predictions_changes = []
        for value in ranges:
            modified_sample = sample.copy()
            modified_sample[feature] = value
            modified_sample_df = pd.DataFrame([modified_sample], columns=X_test_qf.columns)

            prediction_for_this_value = np.zeros(1)
            # 遍历每个模型进行预测
            for model_name, weight in weights_qf.items():
                for i in range(1,6):
                    model_path = f'{folder_path_qf}/{model_name}{i}_new.pkl'
                    model = joblib.load(model_path)
                    prediction_for_this_value += model.predict(modified_sample_df) * weight

            predictions_changes.append((value, prediction_for_this_value[0]))

        # 将结果转换为 DataFrame 并保存到 Excel的不同sheet中
        results_df = pd.DataFrame(predictions_changes, columns=[feature, 'Predicted Yield Strength'])
        results_df.to_excel(writer, sheet_name=feature, index=False)

print("Multi-feature single variable analysis completed and results saved to Excel.")



# 定义文件夹路径
folder_path_kl = "kl_models"
# 权重定义
weights_kl = {'xgboost': 0.5 / 5, 'rf': 0.5 / 5}

# 加载测试集
df_kl = pd.read_excel('kl_models/test_set_new.xlsx', index_col=0)
X_test_kl = df_kl.drop(columns=['Precipitate Distribution', 'Yield_Strength', 'Tensile_Strength (UTS)', '追踪编号'])
y_test_kl = df_kl['Tensile_Strength (UTS)']

# 选择一个样本

# 创建Excel文件以保存结果
with pd.ExcelWriter('kl_multiple_single_variable_analysis.xlsx', engine='openpyxl') as writer:
    for feature, ranges in feature_ranges.items():
        predictions_changes = []
        for value in ranges:
            modified_sample = sample.copy()
            modified_sample[feature] = value
            modified_sample_df = pd.DataFrame([modified_sample], columns=X_test_kl.columns)
            prediction_for_this_value = np.zeros(1)
            # 遍历每个模型进行预测
            for model_name, weight in weights_kl.items():
                for i in range(1,6):
                    model_path = f'{folder_path_kl}/{model_name}{i}_new.pkl'
                    model = joblib.load(model_path)
                    prediction_for_this_value += model.predict(modified_sample_df) * weight

            predictions_changes.append((value, prediction_for_this_value[0]))

        # 将结果转换为 DataFrame 并保存到 Excel的不同sheet中
        results_df = pd.DataFrame(predictions_changes, columns=[feature, 'Predicted Tensile Strength'])
        results_df.to_excel(writer, sheet_name=feature, index=False)

print("Multi-feature single variable analysis for tensile strength completed and results saved to Excel.")


Multi-feature single variable analysis completed and results saved to Excel.
Multi-feature single variable analysis for tensile strength completed and results saved to Excel.
